In [10]:
import scipy.io
import pandas as pd
import os

# Load the .mat file
mat_file_path = '/Users/nagasaivegur/Developer/wizard/IQA/SAUD/sub_votes.mat'
mat_data = scipy.io.loadmat(mat_file_path)

# Extract the 'sub_votes' array from the loaded data
sub_votes = mat_data['sub_votes']

# Convert to a DataFrame for easier manipulation
sub_votes_df = pd.DataFrame(sub_votes[:, 0:2], columns=['Name', 'MOS'])

# Extract only the file name from the full path
sub_votes_df['Name'] = sub_votes_df['Name'].apply(lambda x: x[0])

# Extract the MOS score from the nested array
sub_votes_df['MOS'] = sub_votes_df['MOS'].apply(lambda x: x[0][0])

def normalize_mos_scores(scores):
    min_score = scores.min()
    max_score = scores.max()
    normalized_scores = (scores - min_score) / (max_score - min_score) * 10
    return normalized_scores

mos_scores = sub_votes_df['MOS']
sub_votes_df['MOS'] = normalize_mos_scores(mos_scores).round(4)

# Save the DataFrame to an Excel file
excel_file_path = '/Users/nagasaivegur/Developer/wizard/IQA/SAUD/mos.xlsx'
sub_votes_df.to_excel(excel_file_path, index=False)

print(f"Data saved to {excel_file_path}")

Data saved to /Users/nagasaivegur/Developer/wizard/IQA/SAUD/mos.xlsx


In [4]:
import os
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms
from tqdm import tqdm

# Define paths
image_dir = "/Users/nagasaivegur/Developer/wizard/IQA/SAUD/Enhanced"  # Path to the SAUD image directory


# Image transformations (excluding normalization)
transform = transforms.Compose([
		transforms.Resize((224, 224)),
		transforms.ToTensor(),
])

# Function to compute mean and std
def compute_mean_std(image_dir):
		# Initialize sums for mean and squared sums for std
		mean_sum = torch.zeros(3)
		std_sum = torch.zeros(3)
		num_pixels = 0

		# Loop over all images in the directory
		for root, _, files in os.walk(image_dir):
			for file in files:
				if file.endswith(('.png', '.jpg', '.jpeg')):
					image_path = os.path.join(root, file)
					
					# Open and transform the image
					image = Image.open(image_path).convert('RGB')
					image_tensor = transform(image)
					
					# Sum the mean of each channel (R, G, B)
					mean_sum += image_tensor.mean(dim=[1, 2])
					
					# Sum the squared differences for std computation
					std_sum += image_tensor.pow(2).mean(dim=[1, 2])
					
					# Keep track of the number of pixels processed
					num_pixels += 1
		
		# Final mean and std computation
		mean_r, mean_g, mean_b = mean_sum / num_pixels
		std_r, std_g, std_b = (std_sum / num_pixels - mean_sum / num_pixels ** 2).sqrt()
		
		return mean_r.item(), mean_g.item(), mean_b.item(), std_r.item(), std_g.item(), std_b.item()

# Compute mean and std for the SAUD dataset
mean_r, mean_g, mean_b, std_r, std_g, std_b = compute_mean_std(image_dir)

# Print computed values
print(f"Computed Mean: [{round(mean_r, 3)}, {round(mean_g, 3)}, {round(mean_b, 3)}]")
print(f"Computed Std: [{round(std_r, 3)}, {round(std_g, 3)}, {round(std_b, 3)}]")

Computed Mean: [0.366, 0.424, 0.413]
Computed Std: [0.442, 0.484, 0.477]


In [4]:
import pandas as pd

# Read the mistakenly saved .xlsx file as a CSV
path = "/Users/nagasaivegur/Developer/wizard/IQA/MSAEQA/features_saud.csv"  # Replace with your actual file path

# Read it as a CSV
df = pd.read_csv(path)

df = df.rename(columns={'id': 'Name'})

# Save it back with the correct .csv extension
df.to_csv(path, index=False)



In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load the saved features CSV
features_df = pd.read_csv("/Users/nagasaivegur/Developer/wizard/IQA/MSAEQA/features_saud.csv")

# Load the annotations with MOS values
annotations_df = pd.read_excel("/Users/nagasaivegur/Developer/wizard/IQA/SAUD/mos.xlsx")  # Assuming annotations are in an Excel file

# Merge the features with annotations on the image IDs (filenames + two parent folders)
merged_df = pd.merge(features_df, annotations_df[['Name', 'MOS']])

# Remove rows where MOS values are missing
merged_df = merged_df.dropna(subset=['MOS'])

# save the merged data to a new CSV file
merged_file_path = "/Users/nagasaivegur/Developer/wizard/IQA/MSAEQA/merged_saud_data.csv"
merged_df.to_csv(merged_file_path, index=False)



1000
